# 身份认证与令牌校验

学习目标：读取 Bearer 凭据，用密码哈希检查演示口令，并通过签名、有效期和声明校验识别本地测试用户。

前置知识：HTTP 请求头与状态码、JSON、Python 类型标注、异常处理、依赖注入与 Pydantic 响应模型。

适用版本：Python 3.12、FastAPI 0.141.1、PyJWT 2.14.0、pwdlib 0.3.1，密码哈希使用 argon2-cffi 25.1.0。

环境准备：见 [FastAPI 环境与运行说明](README.md)。

工作目录：content/Web与应用开发/FastAPI。选择课程环境的 Python 3 (ipykernel)，从空内核顺序运行；TestClient 在进程内调用应用，with 关闭客户端。

本章使用虚构用户和演示口令，在内核中生成临时签名密钥与教学令牌；输出只展示校验结果和公开用户信息。重启内核后需重新生成令牌，阅读结束后可关闭内核。

教学令牌用于观察 API 的校验过程；真实应用的登录与令牌签发应按所选身份提供方的协议集成。

本章的口令检查与令牌签发在内核中完成。RFC 9700 禁止使用 OAuth2 资源所有者密码凭据授权，因此这些操作不应被拼成新应用的该授权流程。

## 1 HTTPBearer 先读取凭据

客户端通常把令牌放在 Authorization 请求头中，格式为 Bearer 后跟一个空格和令牌字符串。HTTPBearer 读取这个格式，返回 HTTPAuthorizationCredentials，其中 scheme 是认证方案，credentials 是凭据内容。

先建一个只观察格式的接口。成功读到 Bearer 字符串还不能证明用户身份，后面仍需校验凭据。Bearer 凭据的持有者可以用它发起请求，因此真实网络传输需要 HTTPS 保护。

In [1]:
from typing import Annotated

from fastapi import Depends, FastAPI, HTTPException
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from fastapi.testclient import TestClient

parser_app = FastAPI()
bearer = HTTPBearer()


@parser_app.get("/credential-format")
def read_credential_format(
    credentials: Annotated[HTTPAuthorizationCredentials, Depends(bearer)],
):
    return {"scheme": credentials.scheme}

C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


用一段明确的演示字符串检查解析，再发送缺少凭据的请求。当前 FastAPI 版本中，HTTPBearer 默认会中止缺少凭据的请求，返回 401，并在 WWW-Authenticate 中声明 Bearer。

In [2]:
with TestClient(parser_app) as client:
    parsed = client.get(
        "/credential-format", headers={"Authorization": "Bearer classroom-example"}
    )
    missing = client.get("/credential-format")
assert parsed.json() == {"scheme": "Bearer"}
assert missing.status_code == 401
assert missing.headers["www-authenticate"] == "Bearer"
# 200 只表示格式被读取，不表示 classroom-example 是有效的身份凭据。
print(parsed.status_code, parsed.json())  # 预期：200 {'scheme': 'Bearer'}。
print(missing.status_code, missing.headers["www-authenticate"])  # 预期：401 Bearer。

200 {'scheme': 'Bearer'}
401 Bearer


## 2 用户标识与公开用户信息

凭据校验通过后，需要用其中的用户标识查询本地用户记录。下面先准备一个虚构用户与查询函数；用户名只是记录的查找键，不能因为客户端提交了它就确认身份。

PublicUser 只声明允许返回的 username 和 display_name。这个公开模型会在受保护接口中作为响应模型使用。

In [3]:
from pydantic import BaseModel


class PublicUser(BaseModel):
    username: str
    display_name: str


users = {"alice": {"username": "alice", "display_name": "测试用户 Alice"}}


def find_user(username: str) -> dict[str, str] | None:
    return users.get(username)


local_user = find_user("alice")
assert local_user is not None
assert find_user("nobody") is None
print(PublicUser.model_validate(local_user).model_dump())  # 预期：{'username': 'alice', 'display_name': '测试用户 Alice'}。

{'username': 'alice', 'display_name': '测试用户 Alice'}


## 3 生成密码哈希，再校验口令

密码哈希用于检查提交的口令是否匹配已保存的记录。pwdlib 的 PasswordHash.recommended() 在本版本使用 Argon2；hash 生成保存值，verify 用提交口令和保存值核对，返回布尔结果。

Argon2 的默认哈希过程会生成随机盐（salt），把盐和算法参数保存在编码后的哈希字符串里。同一口令重复哈希通常会得到不同字符串，所以验证时应调用 verify。

下面的口令只用于本地演示。内部用户记录增加 password_hash 字段，展示时只打印匹配结果。

In [4]:
from pwdlib import PasswordHash

password_hasher = PasswordHash.recommended()
demo_password = "classroom-only-password"
users["alice"]["password_hash"] = password_hasher.hash(demo_password)

stored_hash = users["alice"]["password_hash"]
matched = password_hasher.verify(demo_password, stored_hash)
rejected = password_hasher.verify("incorrect-demo-password", stored_hash)
another_hash = password_hasher.hash(demo_password)
assert matched is True and rejected is False
assert password_hasher.verify(demo_password, another_hash) is True
# 两个哈希都匹配同一口令；随机盐使直接比较哈希字符串没有验证意义。
print("正确口令：", matched, "错误口令：", rejected)  # 预期：正确口令： True 错误口令： False。
print("两次哈希不同：", another_hash != stored_hash)  # 预期：两次哈希不同： True。

正确口令： True 错误口令： False
两次哈希不同： True


## 4 签发一枚带有效期的教学 JWT

JWT（JSON Web Token）承载声明（claim），即关于用户和令牌的一组信息。本例生成带签名的 JWT，内容可以被读取，签名用于检查完整性；声明中只放用户标识与校验所需信息。

本章自行要求下列五个声明都存在。它们的必要性由本例的令牌约定确定，不能把本表理解成所有 JWT 都自动具有这些字段。

| 声明 | 中文名称／含义 | 本例的约定 |
| --- | --- | --- |
| sub | 主体标识 | 非空字符串，对应本地用户名 |
| iss | 签发者 | 固定为 classroom-issuer |
| aud | 预期接收方 | 本 API 标识 classroom-api |
| iat | 签发时间 | Unix 纪元起的秒数 |
| exp | 过期时间 | 同样使用秒数，本例设为签发后 10 分钟 |

PyJWT 可以把 UTC datetime 转成令牌使用的时间戳，单位为从 Unix 纪元起经过的秒。HS256 使用共享秘密密钥生成和验证签名；本例每次执行用 secrets 生成 32 字节临时密钥，满足 HS256 的最小密钥长度。

In [5]:
from datetime import datetime, timedelta, timezone
import secrets

import jwt

signing_key = secrets.token_bytes(32)
issuer = "classroom-issuer"
audience = "classroom-api"
now = datetime.now(timezone.utc)
base_claims = {
    "sub": "alice",
    "iss": issuer,
    "aud": audience,
    "iat": now,
    "exp": now + timedelta(minutes=10),
}
# 签发仅在内核内进行；这里先核对本地演示用户的口令。
assert password_hasher.verify(demo_password, users["alice"]["password_hash"])
valid_token = jwt.encode(base_claims, signing_key, algorithm="HS256")
print("教学令牌已生成；声明名称：", sorted(base_claims))  # 预期：教学令牌已生成；声明名称： ['aud', 'exp', 'iat', 'iss', 'sub']。

教学令牌已生成；声明名称： ['aud', 'exp', 'iat', 'iss', 'sub']


## 5 固定校验规则，再信任声明

jwt.decode 在验证签名后按选项检查声明。algorithms 是服务端的允许列表，本例固定为 HS256，不能根据令牌头里的 alg 选择算法。

HTTPBearer 提取出字符串，不代表里面的 sub 已经可信。先划出认证路径，再区分“读取凭据”与“验证后识别用户”。

![从 Bearer 字符串到可信用户身份](image/illustration/12-01-token-validation.svg)

图示：本章认证逻辑示意；JWT 校验包含必需声明、时间与签发者等约束，图不规定库内部逐条检查的实现顺序。

issuer 与 audience 指定本 API 接受的签发者和接收方；require 只要求字段存在，仍须保留签名、时间和声明值校验。

时间声明使用 NumericDate，即从 Unix 纪元起经过的秒数，可以含小数。JSON 布尔值、字符串、数组与对象都不是这个数值；NaN 和无穷大也不符合 JSON 数值要求。下面先写一个小函数检查类型和有限值。type(value) 精确区分 int、float 与 bool，避免把布尔值当作整数接受。

下面先补充 NumericDate 的类型检查，再阅读 jwt.decode 的固定参数；后续用过期、篡改和缺少声明的令牌分别检查拒绝路径。

In [6]:
from math import isfinite


def check_numeric_date(value: object) -> None:
    if type(value) not in (int, float):
        raise jwt.InvalidTokenError("Time claim must be a JSON number")
    if type(value) is float and not isfinite(value):
        raise jwt.InvalidTokenError("Time claim must be finite")

PyJWT 2.14.0 的时间检查包含整数转换：可转换的值不一定符合 JSON 类型要求，容器或无穷大还可能触发 TypeError、OverflowError。下面只在 jwt.decode 调用边界把这两类输入异常转成 InvalidTokenError，再严格检查返回的时间值；默认校验保持开启。

iat、exp 是本例的必需声明；可选的 nbf 表示令牌开始生效的时间，提供时也接受相同的类型检查。空 sub 另按本例约定拒绝。库内部按整数秒进行时间比较，本例的签发代码也生成整数秒；这里的类型检查仍接受标准允许的有限小数。

In [7]:
required_claims = ["sub", "iss", "aud", "iat", "exp"]


def decode_token(token: str) -> dict:
    # 1. 校验签名、算法、签发者、接收者与必需声明，再使用解码结果。
    try:
        claims = jwt.decode(
            token,
            signing_key,
            algorithms=["HS256"],
            issuer=issuer,
            audience=audience,
            options={"require": required_claims},
        )
    except (TypeError, OverflowError) as error:
        raise jwt.InvalidTokenError("Malformed claim value") from error
    # 2. 日期声明的类型检查属于本接口契约，不把异常值当作可信时间。
    for name in ("iat", "exp", "nbf"):
        if name in claims:
            check_numeric_date(claims[name])
    if not claims["sub"]:
        raise jwt.InvalidTokenError("Empty subject")
    return claims


checked_claims = decode_token(valid_token)
assert checked_claims["sub"] == "alice"
assert type(checked_claims["exp"]) is int
# 只展示已校验的主体；不输出完整令牌、签名或密钥。
print("已校验主体：", checked_claims["sub"])  # 预期：已校验主体： alice。

已校验主体： alice


## 6 将校验放入依赖，并过滤响应字段

受保护接口使用一个依赖完成三步：读取 Bearer 凭据、校验 JWT、查询本地用户。令牌无效或用户不存在时返回 401，并附带 WWW-Authenticate: Bearer。

decode_token 将预期的令牌校验失败统一为 InvalidTokenError。依赖只捕获这一类错误并转换为 401；用户查询等其他程序错误仍交给应用的错误处理机制。

In [8]:
def current_user(
    credentials: Annotated[HTTPAuthorizationCredentials, Depends(bearer)],
) -> dict[str, str]:
    authentication_error = HTTPException(
        status_code=401,
        detail="无法验证凭据",
        headers={"WWW-Authenticate": "Bearer"},
    )
    try:
        claims = decode_token(credentials.credentials)
    except jwt.InvalidTokenError as error:
        raise authentication_error from error
    user = find_user(claims["sub"])
    if user is None:
        raise authentication_error
    return user

新建只包含受保护接口的 protected_app，返回依赖得到的用户记录。response_model=PublicUser 会校验并过滤响应，只保留公开字段；即使内部记录有 password_hash，也不会作为这个响应返回。

In [9]:
protected_app = FastAPI()


@protected_app.get("/me", response_model=PublicUser)
def read_me(user: Annotated[dict[str, str], Depends(current_user)]):
    return user


with TestClient(protected_app) as client:
    response = client.get("/me", headers={"Authorization": f"Bearer {valid_token}"})
assert response.status_code == 200
assert response.json() == {"username": "alice", "display_name": "测试用户 Alice"}
assert "password_hash" in users["alice"]
assert "password_hash" not in response.json()
print(response.status_code, response.json())  # 预期：200 {'username': 'alice', 'display_name': '测试用户 Alice'}。

200 {'username': 'alice', 'display_name': '测试用户 Alice'}


## 7 检查缺失、过期和篡改的令牌

令牌签名正确也可能已经过期。构造一个过期时间位于过去的样本，就可以立即观察拒绝行为，无需等待。

本例的签名 JWT 由点分隔为三段。另一个样本只替换签名段的首个字符，保留原有声明，用于检查签名变化是否被拒绝。

In [10]:
# 构造过期令牌与被改动的签名，只使用本章虚构身份和临时密钥。
expired_claims = {**base_claims, "exp": now - timedelta(minutes=1)}
expired_token = jwt.encode(expired_claims, signing_key, algorithm="HS256")
header_part, payload_part, signature_part = valid_token.split(".")
changed_first = "A" if signature_part[0] != "A" else "B"
tampered_token = ".".join(
    (header_part, payload_part, changed_first + signature_part[1:])
)
cases = [
    ("缺失", None),
    ("过期", expired_token),
    ("签名篡改", tampered_token),
    ("格式错误", "not-a-jwt"),
]
# 逐个请求保护端点，观察它们统一返回的认证失败契约。
with TestClient(protected_app) as client:
    for label, token in cases:
        headers = {} if token is None else {"Authorization": f"Bearer {token}"}
        response = client.get("/me", headers=headers)
        assert response.status_code == 401
        assert response.headers["www-authenticate"] == "Bearer"
        # 所有失败都要求重新提供 Bearer 凭据，且没有返回用户信息。
        assert set(response.json()) == {"detail"}
        # 预期：缺失、过期、签名篡改、格式错误四种情况均为 401 Bearer。
        print(label, response.status_code, response.headers["www-authenticate"])

缺失 401 Bearer
过期 401 Bearer
签名篡改 401 Bearer
格式错误 401 Bearer


## 8 检查必要声明与算法限制

字段缺失和字段值错误是两种不同情况。先分别删除一个必要声明，重新用本地密钥签名，确认这些样本即使签名正确也会被拒绝。

In [11]:
with TestClient(protected_app) as client:
    for claim_name in required_claims:
        incomplete = base_claims.copy()
        del incomplete[claim_name]
        token = jwt.encode(incomplete, signing_key, algorithm="HS256")
        response = client.get("/me", headers={"Authorization": f"Bearer {token}"})
        assert response.status_code == 401
        assert response.headers["www-authenticate"] == "Bearer"
        print("缺少", claim_name, response.status_code)  # 预期：依次缺少 sub、iss、aud、iat、exp 时，状态码均为 401。

缺少 sub 401
缺少 iss 401
缺少 aud 401
缺少 iat 401
缺少 exp 401


再保留字段，改变签发者、接收方、主体或时间值。sub 的字符串类型由 PyJWT 检查；空主体按本例规则拒绝，不存在的用户名则在本地查询阶段拒绝。

In [12]:
invalid_values = [
    ("iss", "another-issuer"),
    ("aud", "another-api"),
    ("sub", 123),
    ("sub", ""),
    ("sub", "nobody"),
    ("iat", "not-a-time"),
    ("exp", "not-a-time"),
]
with TestClient(protected_app) as client:
    for field, value in invalid_values:
        changed = {**base_claims, field: value}
        token = jwt.encode(changed, signing_key, algorithm="HS256")
        response = client.get("/me", headers={"Authorization": f"Bearer {token}"})
        assert response.status_code == 401
        assert response.headers["www-authenticate"] == "Bearer"
        print(field, repr(value), response.status_code)  # 预期：逐行显示测试字段及值，七组无效声明均返回 401。

iss 'another-issuer' 401
aud 'another-api' 401
sub 123 401
sub '' 401
sub 'nobody' 401
iat 'not-a-time' 401
exp 'not-a-time' 401


时间声明还要检查类型，不能只测无法转换的字符串。下面分别给 iat、exp、nbf 提供错误类型，样本仍由本地临时密钥签名。数字字符串虽然能转成整数，仍应拒绝；特殊浮点值用于检查解析器接受了非标准 JSON 值时的拒绝边界。

In [13]:
with TestClient(protected_app) as client:
    for field in ("iat", "exp", "nbf"):
        numeric_text = str(int(now.timestamp()) + (600 if field == "exp" else -30))
        wrong_types = [[], {}, True, numeric_text, None]
        wrong_types += [float("nan"), float("inf"), float("-inf")]
        for value in wrong_types:
            token = jwt.encode(
                {**base_claims, field: value}, signing_key, algorithm="HS256"
            )
            response = client.get("/me", headers={"Authorization": f"Bearer {token}"})
            assert response.status_code == 401
            assert response.headers["www-authenticate"] == "Bearer"
            assert set(response.json()) == {"detail"}
        print(field, "的 8 组错误类型均返回 401")  # 预期：依次显示 iat、exp、nbf 的八组错误类型均返回 401。
# TestClient 保留默认的异常传播；若仍有未处理异常，这里会直接失败。

iat 的 8 组错误类型均返回 401
exp 的 8 组错误类型均返回 401


nbf 的 8 组错误类型均返回 401


拒绝错误输入后，再给出合法的整数与小数作为对照，并检查未来签发时间和未来生效时间仍被拒绝。这里用明确位于过去或未来的时间，避免依赖亚秒边界。

In [14]:
past_seconds = int(now.timestamp()) - 30
future_seconds = int(now.timestamp()) + 600
checks = [
    ({"iat": past_seconds, "nbf": past_seconds}, 200),
    ({"iat": past_seconds + 0.5, "exp": future_seconds + 0.5}, 200),
    ({"iat": future_seconds}, 401),
    ({"nbf": future_seconds}, 401),
]
with TestClient(protected_app) as client:
    for changes, expected in checks:
        token = jwt.encode({**base_claims, **changes}, signing_key, algorithm="HS256")
        response = client.get("/me", headers={"Authorization": f"Bearer {token}"})
        assert response.status_code == expected
        if expected == 401:
            assert response.headers["www-authenticate"] == "Bearer"
        else:
            assert response.json() == PublicUser.model_validate(users["alice"]).model_dump()
        print("时间声明对照：", response.status_code)  # 预期：四组对照依次返回 200、200、401、401。

时间声明对照： 200
时间声明对照： 200
时间声明对照： 401
时间声明对照： 401


最后检查允许列表：即使某个令牌使用另一种算法生成了签名，本 API 仍只接受预先配置的 HS256。下面另用满足长度要求的临时密钥生成 HS512 样本。

In [15]:
other_algorithm_token = jwt.encode(
    base_claims, secrets.token_bytes(64), algorithm="HS512"
)
try:
    decode_token(other_algorithm_token)
except jwt.InvalidAlgorithmError:
    print("HS512 不在允许列表中")  # 预期：不允许 HS512 的检查失败，显示此提示。
else:
    raise AssertionError("不允许的算法未被拒绝")

with TestClient(protected_app) as client:
    response = client.get(
        "/me", headers={"Authorization": f"Bearer {other_algorithm_token}"}
    )
assert response.status_code == 401
assert response.headers["www-authenticate"] == "Bearer"
print(response.status_code, response.headers["www-authenticate"])  # 预期：401 Bearer。

HS512 不在允许列表中
401 Bearer


## 本章小结

（1）HTTPBearer 解析请求头；用户身份还需要凭据校验和本地记录查询。

（2）pwdlib 用 Argon2 哈希保存演示口令的校验信息，使用 verify 核对提交值。

（3）JWT 接受条件包括签名、固定算法、时间值及其类型、必要声明，以及预期签发者和接收方；读出内容本身不构成校验。

（4）认证失败返回 401 与 Bearer 认证提示；响应模型只返回公开用户字段。

自查：一个签名正确的 JWT，还可能因哪些条件被 /me 拒绝？密码哈希应该出现在响应或令牌声明中吗？

## 练习

（1）新增虚构用户 bob，用运行时生成的哈希保存另一条演示口令，再为 bob 签发符合本章约定的令牌。验证标准：正确和错误口令分别得到 True、False；/me 返回 bob 的公开资料，结果没有 password_hash。

（2）把正确签发的令牌的 aud 设为另一个 API 标识。验证标准：签名仍由本章密钥生成，但 /me 返回 401 和 WWW-Authenticate: Bearer。

（3）分别构造缺少 exp 与 exp 为过去时间的令牌。验证标准：调用 decode_token 时分别出现 MissingRequiredClaimError 与 ExpiredSignatureError；经 /me 调用时两者都返回 401。

（4）删除 users 中的 alice，再用尚未过期的原令牌调用 /me。验证标准：即使 JWT 校验通过，本地用户已不存在时仍返回 401；练习后恢复用户记录。

（5）给 nbf 设置数字字符串与合法的过去时间数值。验证标准：前者返回 401，后者在其余声明有效时返回 200。

提示：异常类型只在本地观察，保存输出时继续只展示名称、状态码和公开资料；不要打印密钥、完整令牌或密码哈希。令牌过期或内核重启后，从空内核重跑即可生成新的实验输入。

## 参考与引用来源

- **FastAPI 官方文档（fastapi.tiangolo.com）**：[Security Tools](https://fastapi.tiangolo.com/reference/security/#fastapi.security.HTTPBearer)，定位 HTTPBearer、HTTPAuthorizationCredentials、make_authenticate_headers 与 make_not_authenticated_error，支持凭据解析及本版本 401 行为；[Password hashing and JWT](https://fastapi.tiangolo.com/tutorial/security/oauth2-jwt/)，仅采用 About JWT、Hash and verify the passwords、Handle JWT tokens 与 Update the dependencies 中的哈希、签名和用户查找说明；[Response Model](https://fastapi.tiangolo.com/tutorial/response-model/#response-model-parameter)，支持响应字段过滤；[Testing](https://fastapi.tiangolo.com/tutorial/testing/#using-testclient)，支持应用内请求与断言。
- **pwdlib 官方文档（frankie567.github.io）**：[Guide](https://frankie567.github.io/pwdlib/guide/)，定位 recommended、Hash a password 与 Verify a password，支持 Argon2 配置、hash 与 verify 的参数及返回值。
- **argon2-cffi 官方文档（argon2-cffi.readthedocs.io）**：[PasswordHasher API](https://argon2-cffi.readthedocs.io/en/stable/api.html#argon2.PasswordHasher.hash)，定位 hash 的 salt 参数，支持随机盐的生成行为。
- **PyJWT 2.14.0 官方文档（pyjwt.readthedocs.io）**：[Usage Examples](https://pyjwt.readthedocs.io/en/stable/usage.html)，定位 Registered Claim Names、Key Length Validation 与 Requiring Presence of Claims，支持声明、UTC 时间、密钥长度与必填配置；[API Reference](https://pyjwt.readthedocs.io/en/stable/api.html#jwt.decode)，定位 algorithms 警告、jwt.types.Options、verify_sub 与 Exceptions，支持算法允许列表、声明值校验和异常类型。
- **Python 3.12 官方文档（docs.python.org）**：[secrets.token_bytes](https://docs.python.org/3.12/library/secrets.html#secrets.token_bytes)，支持生成指定字节数的临时随机密钥；[type](https://docs.python.org/3.12/library/functions.html#type) 与 [math.isfinite](https://docs.python.org/3.12/library/math.html#math.isfinite) 支持精确类型及有限值检查。
- **RFC Editor（rfc-editor.org）**：[RFC 6750 第 2.1、3、5 节](https://www.rfc-editor.org/rfc/rfc6750.html#section-2.1)，支持 Bearer 请求头、认证提示和传输保护；[RFC 7519 第 3、4.1 节](https://www.rfc-editor.org/rfc/rfc7519.html#section-3)，支持 JWT 表示及注册声明；[RFC 7519 第 2 节](https://www.rfc-editor.org/rfc/rfc7519.html#section-2) 的 NumericDate 与 [RFC 8259 第 6 节](https://www.rfc-editor.org/rfc/rfc8259.html#section-6) 支持时间数值及 JSON 有限数值边界；[RFC 7518 第 3.2 节](https://www.rfc-editor.org/rfc/rfc7518.html#section-3.2)，支持 HMAC 与 HS256 密钥长度；[RFC 9700 第 2.3、2.4 节](https://www.rfc-editor.org/rfc/rfc9700.html#section-2.4)，支持接收方限制与禁止密码凭据授权的边界。
- **Starlette 官方文档（starlette.dev）**：[TestClient](https://starlette.dev/testclient/)，支持本章客户端的 with 使用范围。
- **PyJWT 第一方源码（raw.githubusercontent.com）**：[2.14.0 的 api_jwt.py](https://raw.githubusercontent.com/jpadilla/pyjwt/2.14.0/jwt/api_jwt.py)，定位 decode_complete、_validate_iat、_validate_nbf 与 _validate_exp；支持签名后声明校验、整数转换和异常边界。本章同时核对课程环境安装的对应函数。